# Module 2: Snowflake Postgres — Customer Self-Service Portal

This module extends the EPOWER demo with a **Snowflake Postgres** instance that backs the **"Mein EPOWER"** customer self-service portal — a web application where 20,000 customers manage their energy accounts online.

---

### The Business Case

Every energy retailer needs a digital customer portal. In Germany, where EPOWER operates, regulatory requirements (Marktkommunikation) and customer expectations demand self-service capabilities: meter reading submission, tariff switching, billing inquiries, and program enrollments.

**"Mein EPOWER"** is EPOWER's customer-facing web application. Behind it: a PostgreSQL database handling the transactional workload — logins, form submissions, order processing, and session management for 20,000 customers.

The challenge: **How do you get operational portal data into your analytical platform without building ETL pipelines?** The answer: Snowflake Postgres + pg_lake. The portal writes to Postgres for operations; Postgres writes to Iceberg for analytics; Snowflake reads Iceberg natively. Zero middleware.

---

### Why Postgres?

The portal is a standard web application stack: **React frontend → REST API → PostgreSQL**. This is the most common backend pattern in software — used by millions of applications worldwide. Customers perform transactional operations that require low-latency reads, form validation, and ACID guarantees:

| Portal Feature | OLTP Requirement | Why Not Snowflake? |
|---------------|-----------------|-------------------|
| **Submit meter readings** | Validation (new ≥ previous), INSERT with constraints | Sub-second response needed for UX |
| **Request tariff switch** | Atomic order with status lifecycle (PENDING → CONFIRMED → ACTIVE) | Row-level locking for concurrent status updates |
| **Open service request** | INSERT with auto-categorization | Hundreds of concurrent form submissions |
| **VPP enrollment** | Multi-step signup with rollback on failure | Transaction semantics (all-or-nothing) |
| **Login & sessions** | Concurrent auth, session UPDATEs, CSRF tokens | Millisecond reads for session validation |

These are standard web-app operations — hundreds of concurrent users, sub-second responses, mutable state with transactional consistency. **This is what Postgres does best.** Snowflake is designed for analytical queries over large datasets, not for serving web application backends.

---

### The Operational → Analytical Bridge

The key architectural insight: **not all data in the portal needs to go to Snowflake, and not all data in Snowflake needs to come from the portal.** We separate concerns:

| Data | Lives in Postgres | Flows to Snowflake? | Why? |
|------|------------------|--------------------| -----|
| User sessions & auth | ✅ | ❌ | Ephemeral, no analytical value |
| Meter readings (raw) | ✅ | ❌ | Already in Snowflake via billing pipeline |
| Tariff orders (mutable) | ✅ | ❌ | Status changes frequently, needs row-locking |
| **Portal activity log** | ✅ | **✅ via pg_lake** | Append-only, denormalized, analytically rich |

The `portal_activity_log` is the bridge — every user action (login, reading submission, tariff change, service request) generates one immutable log entry with customer context. This append-only stream is the natural replication target: it never changes after creation, it's time-partitioned, and it contains everything needed for engagement analytics.

---

### Architecture

```
┌───────────────────────────────────────────────────────────────────┐
│  "Mein EPOWER" Portal (Web Application)                           │
│                                                                   │
│   React Frontend → REST API → Snowflake Postgres                  │
│                                                                   │
│  ┌──────────────┐  ┌────────────────┐  ┌──────────────────────┐  │
│  │ portal_users │  │ meter_readings │  │ tariff_orders         │  │
│  │ (sessions,   │  │ (monthly kWh   │  │ service_requests      │  │
│  │  prefs)      │  │  submissions)  │  │ (tickets, VPP enroll) │  │
│  └──────────────┘  └────────────────┘  └──────────────────────┘  │
│                                                                   │
│       Every user action → portal_activity_log (append-only)       │
│                                    │                              │
│                         pg_incremental (1 min)                    │
│                                    ▼                              │
│                         ┌────────────────────┐                    │
│                         │  Iceberg table      │                    │
│                         │  (pg_lake managed)  │                    │
│                         └─────────┬──────────┘                    │
└───────────────────────────────────┼───────────────────────────────┘
                                    │
                         Catalog Integration
                          (auto-refresh 30s)
                                    ▼
┌───────────────────────────────────────────────────────────────────┐
│  SNOWFLAKE (Analytics + AI) — from Module 1                       │
│                                                                   │
│  EPOWER_BRONZE                                                    │
│  ┌─────────────────────────────────────────────────────────────┐  │
│  │ PORTAL_ACTIVITY_LOG (Iceberg table)  ← Near real-time       │  │
│  └──────────────────────────┬──────────────────────────────────┘  │
│                             │                                     │
│                             ▼                                     │
│  EPOWER_GOLD                                                      │
│  ┌─────────────────────────────────────────────────────────────┐  │
│  │ MART_PORTAL_ENGAGEMENT          Daily engagement metrics    │  │
│  └──────────────────────────┬──────────────────────────────────┘  │
│                             ▼                                     │
│  ┌─────────────────────────────────────────────────────────────┐  │
│  │ PORTAL_SEMANTIC_VIEW              Text-to-SQL               │  │
│  └──────────────────────────┬──────────────────────────────────┘  │
│                             ▼                                     │
│  ┌─────────────────────────────────────────────────────────────┐  │
│  │ EPOWER INTELLIGENCE AGENT         + portal_analyst tool     │  │
│  └─────────────────────────────────────────────────────────────┘  │
└───────────────────────────────────────────────────────────────────┘
```

---

### Snowflake Features Introduced

| Feature | What It Is | Role in This Module |
|---------|-----------|-------------------|
| **Snowflake Postgres** | Fully managed PostgreSQL running inside the Snowflake platform. No patching, no backups, no infrastructure. Connects via standard `psql`, JDBC, or any ORM. | The portal's transactional backend — serving the web application |
| **pg_lake** | Open-source Postgres extension (Apache license) that adds native Iceberg table support. `CREATE TABLE ... USING iceberg` makes Postgres the Iceberg catalog. | Creates and manages the Iceberg table that holds the activity log |
| **pg_incremental** | Postgres extension for automated, exactly-once incremental data pipelines. Processes time intervals with guaranteed no duplicates. | Syncs new activity log entries from heap → Iceberg every minute |
| **Catalog Integration** | Snowflake object that connects to an external Iceberg catalog (here: Postgres). Snowflake reads Iceberg metadata directly — no data copying. | The bridge: tells Snowflake where to find the Postgres-managed Iceberg table |
| **Iceberg Tables** | Open table format (Apache Iceberg). Data stored as Parquet files in object storage, metadata managed by a catalog. Readable by any engine. | The data format — Postgres writes it, Snowflake reads it, no lock-in |
| **Auto-Refresh** | Snowflake periodically polls the Iceberg catalog for new snapshots. Configurable interval (default 30s). | Ensures Snowflake always sees the latest portal activity without manual refresh |

---

### What You'll Learn

| Section | What we do |
|---------|------------|
| **§1** Prerequisites | Verify Module 1 is deployed |
| **§2** Postgres Instance | Provision Snowflake Postgres for the portal backend |
| **§3** Portal Schema | Create tables: users, meter readings, tariff orders, service requests, activity log |
| **§4** Seed Portal Data | Generate 60 days of realistic portal activity for 20K customers |
| **§5** pg_lake + Iceberg Sync | Replicate the activity log to Iceberg automatically |
| **§6** Snowflake Catalog Integration | Connect Snowflake to the Postgres-managed Iceberg table |
| **§7** Analytics Model | Create engagement metrics in EPOWER_GOLD |
| **§8** Semantic View + Agent | Add portal_analyst tool to the Intelligence Agent |
| **§9** Verification & Demo | Validate pipeline, test with natural language, demo live sync |

**Runtime**: ~10 minutes | **Prerequisite**: Module 1 (`epower_hol.ipynb`) must be deployed

## 1. Prerequisites

This module requires Module 1 (`epower_hol.ipynb`) to be fully deployed — we need the customer dimension and product data.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;
USE DATABASE EPOWER_DEMO;

SELECT 'CUSTOMER_DIM' AS required_object, COUNT(*) AS rows FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
UNION ALL SELECT 'PRODUCT_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
UNION ALL SELECT 'CUSTOMER_PRODUCTS', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_PRODUCTS
ORDER BY required_object;

## 2. Create Snowflake Postgres Instance

We provision a Snowflake Postgres instance to serve as the **portal backend** — the transactional database behind the "Mein EPOWER" web application.

&nbsp;

> **Snowflake Feature:** Snowflake Postgres is fully managed PostgreSQL. The portal's web application connects to it like any standard Postgres database — via `psql`, JDBC, or any ORM (Django, Rails, SQLAlchemy). No infrastructure to manage.

In [ ]:
%%sql
CREATE POSTGRES INSTANCE IF NOT EXISTS EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL
    POSTGRES_SIZE = 'XSMALL'
    AUTO_SUSPEND = 600
    AUTO_RESUME = TRUE
    COMMENT = 'Mein EPOWER — customer self-service portal backend (20K customers)';

In [ ]:
%%sql
DESCRIBE POSTGRES INSTANCE EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL;

## 3. Portal Schema

The portal database has **4 operational tables** (the web app's backend) plus **1 activity log** (append-only, replicated to Snowflake):

| Table | Purpose | Why Postgres |
|-------|---------|-------------|
| **portal_users** | Auth, sessions, preferences | Concurrent logins, frequent session UPDATEs |
| **meter_readings** | Monthly kWh submissions | INSERT with validation (new ≥ previous reading) |
| **tariff_orders** | Tariff switch requests | Status lifecycle: PENDING → CONFIRMED → ACTIVE |
| **service_requests** | Support tickets + VPP enrollments | INSERT with auto-categorization |
| **portal_activity_log** | Every user action (append-only) | → Replicated to Snowflake via pg_lake |

The first 4 tables are classic OLTP — mutable state, concurrent access, constraint enforcement. The activity log captures all user interactions as an immutable stream, which is the natural replication target for analytics.

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$

CREATE TABLE IF NOT EXISTS portal_users (
    customer_key       INT PRIMARY KEY,
    email              TEXT NOT NULL UNIQUE,
    display_name       TEXT NOT NULL,
    registered_at      TIMESTAMPTZ DEFAULT now(),
    last_login_at      TIMESTAMPTZ,
    login_count        INT DEFAULT 0,
    preferred_language TEXT DEFAULT 'de',
    notifications_enabled BOOLEAN DEFAULT TRUE
);

CREATE TABLE IF NOT EXISTS meter_readings (
    reading_id         BIGSERIAL PRIMARY KEY,
    customer_key       INT NOT NULL REFERENCES portal_users(customer_key),
    reading_date       DATE NOT NULL,
    meter_type         TEXT NOT NULL,
    reading_kwh        INT NOT NULL,
    submitted_at       TIMESTAMPTZ DEFAULT now(),
    source             TEXT DEFAULT 'PORTAL',
    CONSTRAINT valid_meter_type CHECK (meter_type IN ('ELECTRICITY', 'GAS'))
);

CREATE TABLE IF NOT EXISTS tariff_orders (
    order_id           BIGSERIAL PRIMARY KEY,
    customer_key       INT NOT NULL REFERENCES portal_users(customer_key),
    order_type         TEXT NOT NULL,
    current_product    TEXT,
    requested_product  TEXT NOT NULL,
    status             TEXT DEFAULT 'PENDING',
    created_at         TIMESTAMPTZ DEFAULT now(),
    confirmed_at       TIMESTAMPTZ,
    effective_date     DATE,
    CONSTRAINT valid_status CHECK (status IN ('PENDING', 'CONFIRMED', 'ACTIVE', 'CANCELLED'))
);

CREATE TABLE IF NOT EXISTS service_requests (
    request_id         BIGSERIAL PRIMARY KEY,
    customer_key       INT NOT NULL REFERENCES portal_users(customer_key),
    request_type       TEXT NOT NULL,
    subject            TEXT NOT NULL,
    description        TEXT,
    status             TEXT DEFAULT 'OPEN',
    priority           TEXT DEFAULT 'NORMAL',
    created_at         TIMESTAMPTZ DEFAULT now(),
    resolved_at        TIMESTAMPTZ,
    CONSTRAINT valid_request_type CHECK (request_type IN ('SUPPORT', 'COMPLAINT', 'VPP_ENROLLMENT', 'VPP_OPTOUT', 'BILLING_INQUIRY', 'MOVE'))
);

-- Activity log: append-only stream of all portal events → replicated to Snowflake
CREATE TABLE IF NOT EXISTS portal_activity_log (
    activity_id        BIGSERIAL PRIMARY KEY,
    customer_key       INT NOT NULL,
    event_time         TIMESTAMPTZ DEFAULT now(),
    event_type         TEXT NOT NULL,
    event_detail       JSONB DEFAULT '{}',
    city               TEXT,
    region             TEXT,
    customer_type      TEXT
);

CREATE INDEX IF NOT EXISTS idx_users_email ON portal_users(email);
CREATE INDEX IF NOT EXISTS idx_readings_customer ON meter_readings(customer_key, reading_date);
CREATE INDEX IF NOT EXISTS idx_orders_status ON tariff_orders(status) WHERE status = 'PENDING';
CREATE INDEX IF NOT EXISTS idx_requests_open ON service_requests(status) WHERE status = 'OPEN';
CREATE INDEX IF NOT EXISTS idx_activity_time ON portal_activity_log(event_time);

$$;

## 4. Seed Portal Data

We generate **60 days** of realistic portal activity for EPOWER's 20,000 customers. Not every customer uses the portal equally — we model realistic engagement:

| Segment | % of Customers | Portal Activity |
|---------|---------------|----------------|
| **Power users** | 10% | Login weekly, submit readings on time, switch tariffs |
| **Regular** | 40% | Login monthly, submit readings |
| **Occasional** | 30% | Login every few months, mostly for billing |
| **Inactive** | 20% | Registered but rarely log in |

Each portal action (login, meter reading, tariff order, service request) produces one entry in the `portal_activity_log`.

&nbsp;

> **Data volume:** ~60K activity log entries over 60 days (realistic for 20K customers with mixed engagement).

In [ ]:
# Load customer data from Snowflake to seed the portal
customers_df = session.sql("""
    SELECT customer_key, customer_name, city, state AS region, customer_type
    FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
    ORDER BY customer_key
""").collect()

products_df = session.sql("""
    SELECT product_key, product_name, category_name
    FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
""").collect()

print(f"Loaded {len(customers_df)} customers and {len(products_df)} products")

In [ ]:
# Register portal users (all 20K customers get accounts)
import random

print("Registering portal users...")
user_values = []
for c in customers_df:
    ck = c['CUSTOMER_KEY']
    name = c['CUSTOMER_NAME'].replace("'", "''")
    email = f"kunde{ck}@epower-portal.de"
    days_ago = random.randint(60, 365)
    user_values.append(f"({ck}, '{email}', '{name}', now() - interval '{days_ago} days', 'de', TRUE)")

for i in range(0, len(user_values), 500):
    batch = user_values[i:i+500]
    sql = f"INSERT INTO portal_users (customer_key, email, display_name, registered_at, preferred_language, notifications_enabled) VALUES {', '.join(batch)} ON CONFLICT (customer_key) DO NOTHING;"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $${sql}$$").collect()

print(f"✓ Registered {len(customers_df)} portal users")

In [ ]:
# Generate 60 days of portal activity
from datetime import datetime, timedelta

product_names = [p['PRODUCT_NAME'] for p in products_df]
electricity_products = [p['PRODUCT_NAME'] for p in products_df if p['CATEGORY_NAME'] in ('Strom',)]
request_types = ['SUPPORT', 'COMPLAINT', 'VPP_ENROLLMENT', 'VPP_OPTOUT', 'BILLING_INQUIRY', 'MOVE']
request_subjects = {
    'SUPPORT': ['Zählerstand korrigieren', 'Login-Probleme', 'App funktioniert nicht', 'Abschlag ändern'],
    'COMPLAINT': ['Rechnung unklar', 'Zu hoher Abschlag', 'Mahnung trotz Zahlung', 'Kein Rückruf erhalten'],
    'VPP_ENROLLMENT': ['ePulse VPP Anmeldung', 'VPP Programm beitreten'],
    'VPP_OPTOUT': ['VPP Programm kündigen', 'ePulse deaktivieren'],
    'BILLING_INQUIRY': ['Rechnung nachfragen', 'Gutschrift fehlt', 'Zahlungsnachweis'],
    'MOVE': ['Umzug melden', 'Adresse ändern', 'Vertrag mitnehmen']
}

# Assign engagement levels
power_users = set(random.sample(range(len(customers_df)), int(len(customers_df) * 0.10)))
regular_users = set(random.sample([i for i in range(len(customers_df)) if i not in power_users], int(len(customers_df) * 0.40)))
occasional_users = set(random.sample([i for i in range(len(customers_df)) if i not in power_users and i not in regular_users], int(len(customers_df) * 0.30)))

activity_values = []
reading_values = []
order_values = []
request_values = []

print("Generating 60 days of portal activity...")

for day_offset in range(60, 0, -1):
    if day_offset % 10 == 0:
        print(f"  Day -{day_offset}")

    for idx, c in enumerate(customers_df):
        # Determine if this customer acts today based on engagement level
        if idx in power_users:
            acts_today = random.random() < 0.15  # ~1x/week
        elif idx in regular_users:
            acts_today = random.random() < 0.05  # ~1.5x/month
        elif idx in occasional_users:
            acts_today = random.random() < 0.015  # ~1x every 2 months
        else:
            acts_today = random.random() < 0.003  # rarely

        if not acts_today:
            continue

        ck = c['CUSTOMER_KEY']
        city = c['CITY'].replace("'", "''") if c['CITY'] else 'Unknown'
        region = c['REGION'].replace("'", "''") if c['REGION'] else 'Unknown'
        ctype = c['CUSTOMER_TYPE'].replace("'", "''") if c['CUSTOMER_TYPE'] else 'Unknown'
        ts = f"now() - interval '{day_offset} days' + interval '{random.randint(6,22)} hours {random.randint(0,59)} minutes'"

        # Choose action
        action = random.choices(
            ['LOGIN', 'METER_READING', 'TARIFF_CHANGE', 'SERVICE_REQUEST'],
            weights=[50, 25, 10, 15]
        )[0]

        if action == 'LOGIN':
            detail = '{"action": "login"}'
        elif action == 'METER_READING':
            meter_type = random.choice(['ELECTRICITY', 'GAS'])
            kwh = random.randint(2000, 35000)
            detail = f'{{"meter_type": "{meter_type}", "reading_kwh": {kwh}}}'
            reading_values.append(f"({ck}, CURRENT_DATE - {day_offset}, '{meter_type}', {kwh}, {ts}, 'PORTAL')")
        elif action == 'TARIFF_CHANGE':
            new_product = random.choice(electricity_products) if electricity_products else 'Ökostrom 100%'
            detail = f'{{"requested_product": "{new_product}"}}'
            status = random.choice(['PENDING', 'CONFIRMED', 'ACTIVE', 'CANCELLED'])
            order_values.append(f"({ck}, 'TARIFF_SWITCH', NULL, '{new_product}', '{status}', {ts}, " +
                (f"{ts} + interval '2 days'" if status in ('CONFIRMED','ACTIVE') else "NULL") + f", CURRENT_DATE - {day_offset} + 30)")
        elif action == 'SERVICE_REQUEST':
            req_type = random.choice(request_types)
            subject = random.choice(request_subjects[req_type]).replace("'", "''")
            detail = f'{{"request_type": "{req_type}", "subject": "{subject}"}}'
            req_status = random.choices(['OPEN', 'CLOSED'], weights=[30, 70])[0]
            request_values.append(f"({ck}, '{req_type}', '{subject}', NULL, '{req_status}', 'NORMAL', {ts}, " +
                (f"{ts} + interval '{random.randint(1,72)} hours'" if req_status == 'CLOSED' else "NULL") + ")")

        activity_values.append(f"({ck}, {ts}, '{action}', '{detail}', '{city}', '{region}', '{ctype}')")

print(f"\n  Activities: {len(activity_values):,}")
print(f"  Meter readings: {len(reading_values):,}")
print(f"  Tariff orders: {len(order_values):,}")
print(f"  Service requests: {len(request_values):,}")

In [ ]:
# Bulk insert all generated data into Postgres
print("Inserting portal activity log...")
for i in range(0, len(activity_values), 500):
    batch = activity_values[i:i+500]
    sql = f"INSERT INTO portal_activity_log (customer_key, event_time, event_type, event_detail, city, region, customer_type) VALUES {', '.join(batch)};"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $${sql}$$").collect()

print("Inserting meter readings...")
for i in range(0, len(reading_values), 500):
    batch = reading_values[i:i+500]
    sql = f"INSERT INTO meter_readings (customer_key, reading_date, meter_type, reading_kwh, submitted_at, source) VALUES {', '.join(batch)};"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $${sql}$$").collect()

print("Inserting tariff orders...")
for i in range(0, len(order_values), 500):
    batch = order_values[i:i+500]
    sql = f"INSERT INTO tariff_orders (customer_key, order_type, current_product, requested_product, status, created_at, confirmed_at, effective_date) VALUES {', '.join(batch)};"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $${sql}$$").collect()

print("Inserting service requests...")
for i in range(0, len(request_values), 500):
    batch = request_values[i:i+500]
    sql = f"INSERT INTO service_requests (customer_key, request_type, subject, description, status, priority, created_at, resolved_at) VALUES {', '.join(batch)};"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $${sql}$$").collect()

print(f"\n✓ All portal data loaded")

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$
    SELECT 'portal_users' AS table_name, count(*) AS rows FROM portal_users
    UNION ALL SELECT 'meter_readings', count(*) FROM meter_readings
    UNION ALL SELECT 'tariff_orders', count(*) FROM tariff_orders
    UNION ALL SELECT 'service_requests', count(*) FROM service_requests
    UNION ALL SELECT 'portal_activity_log', count(*) FROM portal_activity_log
    ORDER BY table_name;
$$;

## 5. pg_lake + Iceberg Sync

We replicate the `portal_activity_log` to Snowflake via pg_lake. This table is append-only (one row per user action) and time-partitioned — ideal for Iceberg sync.

The operational tables (`portal_users`, `meter_readings`, `tariff_orders`, `service_requests`) stay in Postgres — they have mutable state and serve the web application directly. The activity log is the **analytics export**: it captures everything the portal does in a denormalized, analysis-friendly format.

&nbsp;

> **Zero-ETL:** No Kafka, no Fivetran, no Airbyte. Postgres writes to Iceberg via pg_lake, Snowflake reads from Iceberg. Data never leaves object storage.

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$

CREATE EXTENSION IF NOT EXISTS pg_lake CASCADE;
CREATE EXTENSION IF NOT EXISTS pg_cron;
CREATE EXTENSION IF NOT EXISTS pg_incremental;

CREATE TABLE IF NOT EXISTS portal_activity_log_iceberg (
    activity_id        BIGINT,
    customer_key       INT,
    event_time         TIMESTAMPTZ,
    event_type         TEXT,
    event_detail       JSONB,
    city               TEXT,
    region             TEXT,
    customer_type      TEXT
) USING iceberg;

$$;

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$

SELECT incremental.create_time_interval_pipeline(
    pipeline_name      := 'sync_portal_activity_to_iceberg',
    time_interval      := '1 minute',
    source_table_name  := 'portal_activity_log',
    start_time         := (SELECT min(event_time) FROM portal_activity_log),
    command            := $inner$
        INSERT INTO portal_activity_log_iceberg
        SELECT activity_id, customer_key, event_time, event_type,
               event_detail, city, region, customer_type
        FROM portal_activity_log
        WHERE event_time >= $1 AND event_time < $2
    $inner$
);

$$;

In [ ]:
%%sql
-- Verify backfill completed
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$
    SELECT
        (SELECT count(*) FROM portal_activity_log) AS heap_rows,
        (SELECT count(*) FROM portal_activity_log_iceberg) AS iceberg_rows;
$$;

## 6. Snowflake Catalog Integration

Connect Snowflake to the Postgres-managed Iceberg table. After this, portal activity data is queryable in Snowflake — with auto-refresh every 30 seconds.

In [ ]:
%%sql
CREATE OR REPLACE CATALOG INTEGRATION EPOWER_DEMO.EPOWER_OPS.PORTAL_POSTGRES_CATALOG
  CATALOG_SOURCE    = SNOWFLAKE_POSTGRES
  TABLE_FORMAT      = ICEBERG
  CATALOG_NAMESPACE = 'public'
  REST_CONFIG = (
    POSTGRES_INSTANCE      = 'EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL'
    CATALOG_NAME           = 'postgres'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  ENABLED = TRUE;

In [ ]:
%%sql
CREATE OR REPLACE ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    CATALOG = 'EPOWER_DEMO.EPOWER_OPS.PORTAL_POSTGRES_CATALOG'
    CATALOG_TABLE_NAME = 'portal_activity_log_iceberg';

ALTER ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    SET AUTO_REFRESH = TRUE;

In [ ]:
%%sql
-- Verify Snowflake can read the portal data from Postgres via Iceberg
SELECT
    count(*) AS total_events,
    count(DISTINCT customer_key) AS unique_customers,
    min(event_time) AS earliest,
    max(event_time) AS latest,
    count(CASE WHEN event_type = 'LOGIN' THEN 1 END) AS logins,
    count(CASE WHEN event_type = 'METER_READING' THEN 1 END) AS readings,
    count(CASE WHEN event_type = 'TARIFF_CHANGE' THEN 1 END) AS tariff_changes,
    count(CASE WHEN event_type = 'SERVICE_REQUEST' THEN 1 END) AS service_requests
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

## 7. Analytics Model

We create a Gold-layer mart that aggregates portal activity into **daily engagement metrics** by region and customer type — answering questions like "How active is our portal?" and "Which regions have low digital adoption?"

In [ ]:
%%sql
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT AS
SELECT
    DATE_TRUNC('DAY', event_time)::DATE AS activity_date,
    region,
    customer_type,
    event_type,
    COUNT(*) AS event_count,
    COUNT(DISTINCT customer_key) AS unique_customers
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
GROUP BY 1, 2, 3, 4
ORDER BY activity_date DESC, region, event_type;

In [ ]:
%%sql
SELECT activity_date, region, event_type, event_count, unique_customers
FROM EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT
ORDER BY activity_date DESC
LIMIT 20;

## 8. Semantic View + Agent Update

We create a **PORTAL_SEMANTIC_VIEW** and add a `portal_analyst` tool to the Intelligence Agent — enabling natural language questions about portal engagement.

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW
  COMMENT = 'Customer portal engagement analytics — sourced from Snowflake Postgres via pg_lake'
AS
TABLES (
  ENGAGEMENT REFERENCES EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT (
    PRIMARY KEY (ACTIVITY_DATE, REGION, CUSTOMER_TYPE, EVENT_TYPE)
    FACTS (
      EVENT_COUNT COMMENT 'Number of portal events'
        SYNONYMS ('events', 'activities', 'actions', 'Aktionen', 'Aktivitäten'),
      UNIQUE_CUSTOMERS COMMENT 'Distinct customers who performed this event type'
        SYNONYMS ('active users', 'aktive Nutzer', 'unique users', 'DAU')
    )
    DIMENSIONS (
      ACTIVITY_DATE COMMENT 'Date of portal activity'
        SYNONYMS ('date', 'day', 'Datum', 'Tag'),
      REGION COMMENT 'German geographic region (Nord, Süd, West, Ost)'
        SYNONYMS ('region', 'Gebiet', 'Bundesland'),
      CUSTOMER_TYPE COMMENT 'Customer segment (Privatkunde, Kleingewerbe, Gewerbekunde)'
        SYNONYMS ('segment', 'Kundentyp', 'customer segment'),
      EVENT_TYPE COMMENT 'Type of portal action: LOGIN, METER_READING, TARIFF_CHANGE, SERVICE_REQUEST'
        SYNONYMS ('action type', 'Aktionstyp', 'event', 'Ereignis')
    )
  )
);

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, customer portal engagement, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export → vpp_telemetry_analyst
    - Portal activity, digital engagement, meter readings, tariff changes, logins → portal_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: portal_analyst, description: "Customer portal engagement: logins, meter readings, tariff changes, service requests, VPP enrollments. Data from Snowflake Postgres via pg_lake."}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  portal_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## 9. Verification & Demo

### Demo Questions for the Agent

| # | Question | What it tests |
|---|----------|---------------|
| 1 | *"How many customers used the portal this week?"* | Basic engagement metric |
| 2 | *"Welche Region hat die höchste Portal-Nutzung?"* | Regional comparison (German) |
| 3 | *"Show me the trend of meter reading submissions over the last 30 days"* | Time-series + charting |
| 4 | *"What are the most popular tariff switches?"* | Tariff change analysis |
| 5 | *"Compare portal engagement between residential and business customers"* | Segment comparison |
| 6 | *"Which customers submit meter readings but have never changed their tariff?"* | **Cross-tool**: portal + sales |

&nbsp;

> **Presenter tip:** Question 6 combines portal data (from Postgres) with sales data (from Snowflake-native tables) — demonstrating the unified platform value.

### Live Demo: Real-Time Sync

Simulate a customer using the portal RIGHT NOW, then watch the data appear in Snowflake within 30-60 seconds.

In [ ]:
%%sql
-- Simulate 50 customers using the portal RIGHT NOW
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.MEIN_EPOWER_PORTAL $$
    INSERT INTO portal_activity_log (customer_key, event_time, event_type, event_detail, city, region, customer_type)
    SELECT
        customer_key,
        now(),
        (ARRAY['LOGIN', 'METER_READING', 'TARIFF_CHANGE', 'SERVICE_REQUEST'])[1 + (random() * 3)::int],
        '{"source": "live_demo"}',
        'Hamburg',
        'Nord',
        'Privatkunde'
    FROM portal_users
    ORDER BY random()
    LIMIT 50;
$$;
-- Wait ~60 seconds, then run the next cell to verify arrival in Snowflake

In [ ]:
%%sql
-- Verify real-time data arrival (run ~60s after previous cell)
SELECT
    MAX(event_time) AS most_recent_event,
    DATEDIFF('second', MAX(event_time), CURRENT_TIMESTAMP()) AS seconds_ago,
    COUNT(*) AS total_rows
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

---

## Summary

In this module you built a complete **web application backend → analytics** pipeline:

| What | How |
|------|-----|
| **Portal backend** | Snowflake Postgres — users, meter readings, tariff orders, service requests |
| **Zero-ETL replication** | pg_lake + pg_incremental → Iceberg (no middleware) |
| **Near real-time** | Auto-refresh every 30 seconds |
| **Analytics** | Gold-layer engagement metrics by region, segment, and action type |
| **AI-ready** | Semantic View + Cortex Agent — queryable in natural language |

**The key insight:** The portal's web application needs Postgres for what web apps always need — low-latency CRUD, session management, form validation, transactional consistency. Snowflake Postgres provides this as a managed service. The analytics-relevant activity stream flows to Snowflake automatically via open standards (Iceberg) — no ETL pipeline to build or maintain.

---

*EPOWER Module 2 — Snowflake Postgres + pg_lake — Powered by Snowflake*

# Module 2: Snowflake Postgres — VPP Command & Control

This module extends the EPOWER demo with a **Snowflake Postgres** instance that serves as the **operational command scheduler** for the ePulse Virtual Power Plant. While Module 1 (`epower_hol.ipynb`) built the analytical layer — dbt pipelines, Semantic Views, and the Intelligence Agent — this module adds the **transactional system** that manages VPP battery schedules.

### Why Postgres?

EPOWER's VPP coordinates ~4,050 home batteries. Every day when day-ahead prices are published (~14:00 CET), a **batch scheduler** calculates optimal charge/discharge schedules for each device and writes them to the database. Each gateway then polls for its own commands every 15 minutes, executes them, and reports the result. This is a classic OLTP workload:

| Requirement | Why Snowflake Alone Can't Do It |
|-------------|--------------------------------|
| **Atomic batch writes** | Write 97K commands (4,050 devices × 24 hours) in one transaction — all or nothing |
| **Sub-second point reads** | Each gateway polls for its own scheduled commands by gateway_id |
| **Concurrent state updates** | Device reports results while monitoring processes check for timeouts |
| **Mutable device state** | Continuous SOC and status updates to the device registry |

The answer: **Snowflake Postgres for operations, Snowflake for analytics** — connected via **pg_lake** (Iceberg).

### How It Works

```
Daily at 14:00 CET:
  Scheduler reads prices → calculates schedules → writes 97K commands (one transaction)

Every 15 minutes:
  Gateway: SELECT ... WHERE gateway_id = 'GW-142' AND state = 'SCHEDULED'
  Gateway picks up command → executes → UPDATE state = 'EXECUTED' (or 'FAILED')
  Log entry written (append-only audit trail)

Concurrently:
  Monitoring process: find SCHEDULED commands older than 20 min → mark as TIMED_OUT
  Billing process: query EXECUTED commands for revenue reconciliation
```

The key ACID requirements: (1) batch writes are atomic, (2) device updates and monitoring don't conflict via row-level locking, (3) billing reads a consistent snapshot.

### Architecture

```
┌────────────────────────────────────────────────────────────────────────┐
│  SNOWFLAKE POSTGRES (ePulse Command Scheduler)                         │
│                                                                        │
│  ┌────────────┐   ┌──────────────────┐   ┌─────────────────────────┐  │
│  │  devices   │   │dispatch_commands │   │ command_execution_log   │  │
│  │  (mutable  │   │  (state machine) │   │  (append-only audit)    │  │
│  │   state)   │   │  SCHEDULED →     │   │                         │  │
│  │            │   │  EXECUTED/FAILED  │   │                         │  │
│  └────────────┘   └──────────────────┘   └────────────┬────────────┘  │
│                                                        │               │
│                                           pg_incremental (1 min)       │
│                                                        ▼               │
│                                            ┌───────────────────────┐   │
│                                            │  Iceberg table        │   │
│                                            │  (pg_lake managed)    │   │
│                                            └───────────┬───────────┘   │
└────────────────────────────────────────────────────────┼───────────────┘
                                                         │
                                             Catalog Integration
                                             (auto-refresh 30s)
                                                         ▼
┌────────────────────────────────────────────────────────────────────────┐
│  SNOWFLAKE (Analytics + AI) — from Module 1                            │
│                                                                        │
│  EPOWER_BRONZE                                                         │
│  ┌─────────────────────────────────────┐                               │
│  │ VPP_COMMAND_EXECUTION_LOG (Iceberg)  │  ← Near real-time            │
│  └───────────────────┬─────────────────┘                               │
│                      │ dbt                                              │
│                      ▼                                                  │
│  EPOWER_GOLD                                                           │
│  ┌─────────────────────────────────────┐                               │
│  │ mart_vpp_dispatch_performance       │  Success rates, latency       │
│  │ mart_vpp_fleet_health               │  Device status, firmware      │
│  └───────────────────┬─────────────────┘                               │
│                      ▼                                                  │
│  ┌─────────────────────────────────────┐                               │
│  │ VPP_DISPATCH_SEMANTIC_VIEW          │  Text-to-SQL                  │
│  └───────────────────┬─────────────────┘                               │
│                      ▼                                                  │
│  ┌─────────────────────────────────────┐                               │
│  │ EPOWER INTELLIGENCE AGENT           │  + dispatch_analyst tool      │
│  └─────────────────────────────────────┘                               │
└────────────────────────────────────────────────────────────────────────┘
```

### What You'll Learn

| Section | What we do |
|---------|------------|
| **§1** Prerequisites Check | Verify Module 1 is deployed |
| **§2** Create Postgres Instance | Provision Snowflake Postgres for VPP dispatch |
| **§3** Operational Schema | Create the dispatch engine tables (devices, commands, execution log, alerts) |
| **§4** Seed Dispatch Data | Generate realistic dispatch history (60 days, price-reactive) |
| **§5** pg_lake + Iceberg Sync | Set up pg_lake, create Iceberg tables, automate sync with pg_incremental |
| **§6** Snowflake Catalog Integration | Connect Snowflake to Postgres-managed Iceberg tables |
| **§7** dbt Models | Add dispatch analytics models (Bronze → Gold) |
| **§8** Semantic View + Agent Update | Create VPP_DISPATCH_SEMANTIC_VIEW and add tool to the agent |
| **§9** Verification & Demo Questions | Validate pipeline and test with natural language questions |

**Runtime**: ~10 minutes | **Prerequisite**: Module 1 (`epower_hol.ipynb`) must be deployed

---

### Snowflake Features Introduced in This Module

| Feature | Role |
|---------|------|
| **Snowflake Postgres** | Managed PostgreSQL for OLTP workloads |
| **pg_lake** | Create and manage Iceberg tables directly from Postgres |
| **pg_incremental** | Automated, exactly-once data sync from heap to Iceberg |
| **Catalog Integration** | Snowflake reads Postgres-managed Iceberg metadata natively |
| **Iceberg Tables** | Open table format — no ETL, no data movement tooling |
| **Auto-Refresh** | Continuous polling for Iceberg metadata changes (configurable interval) |

## 1. Prerequisites Check

This module requires a fully deployed Module 1. Let's verify the core objects exist.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;
USE DATABASE EPOWER_DEMO;

SELECT 'EPULSE_DEVICES' AS required_object, COUNT(*) AS rows FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES
UNION ALL SELECT 'RAW_DAY_AHEAD_PRICES', COUNT(*) FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES
UNION ALL SELECT 'MART_DAY_AHEAD_PRICES', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.MART_DAY_AHEAD_PRICES
UNION ALL SELECT 'CUSTOMER_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
ORDER BY required_object;

## 2. Create Snowflake Postgres Instance

We provision a Snowflake Postgres instance that will serve as the **ePulse Dispatch Engine** — the operational brain that sends charge/discharge commands to 4,050 home batteries in real time.

| Parameter | Value | Rationale |
|-----------|-------|----------|
| **Instance name** | `EPULSE_DISPATCH` | Descriptive, matches the VPP use case |
| **Size** | `xsmall` | Sufficient for demo workload (4K devices, 389K commands/day) |
| **Auto-suspend** | 10 min | Saves credits during demo pauses |

&nbsp;

> **Snowflake Feature:** Snowflake Postgres is fully managed PostgreSQL — no patching, no backups to configure, no infrastructure to maintain. It runs inside the Snowflake platform with native integration to Iceberg via pg_lake.

In [ ]:
%%sql
CREATE POSTGRES INSTANCE IF NOT EXISTS EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH
    POSTGRES_SIZE = 'XSMALL'
    AUTO_SUSPEND = 600
    AUTO_RESUME = TRUE
    COMMENT = 'ePulse VPP Dispatch Engine — OLTP command scheduling for 4,050 battery devices';

In [ ]:
%%sql
SHOW POSTGRES INSTANCES IN SCHEMA EPOWER_DEMO.EPOWER_OPS;
DESCRIBE POSTGRES INSTANCE EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH;

## 3. Operational Schema (Postgres)

Now we create the command scheduler schema inside Postgres. This is the **OLTP layer** — designed for transactional command management:

| Table | Purpose | Access Pattern |
|-------|---------|---------------|
| **devices** | Device registry + mutable state (SOC, firmware, online status) | Point reads/updates by gateway_id |
| **dispatch_commands** | Scheduled commands with 2-state lifecycle: SCHEDULED → EXECUTED/FAILED | Atomic bulk insert (daily), point-read by gateway, concurrent status updates |
| **command_execution_log** | Append-only audit trail (one entry per completed command) | Insert on completion, replicated to Snowflake |
| **dispatch_alerts** | SLA violations (timeouts, offline devices) | Insert on threshold breach, query open alerts |

### Command Lifecycle (2-state model)

```
Scheduler writes command (state = SCHEDULED)
     │
     │  gateway polls: SELECT ... WHERE gateway_id = X AND state = 'SCHEDULED'
     ▼
Gateway picks up command, executes battery action
     │
     │  gateway reports: UPDATE state = 'EXECUTED' (or 'FAILED'), writes log entry
     ▼
Done. Log entry flows to Snowflake via pg_lake.
```

The `command_execution_log` is what we replicate to Snowflake — it's append-only and time-partitioned, making it ideal for Iceberg sync. Postgres keeps only the hot operational window; full history lives in Iceberg.

&nbsp;

> **Design principle:** Postgres handles the transactional lifecycle (atomic writes, concurrent updates, mutable state). Snowflake handles analytics over the full history.

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$

-- Device registry: mutable state updated by gateways on every poll
CREATE TABLE IF NOT EXISTS devices (
    gateway_id         TEXT PRIMARY KEY,
    customer_key       INT NOT NULL,
    firmware_version   TEXT DEFAULT '2.4.1',
    last_seen_at       TIMESTAMPTZ DEFAULT now(),
    status             TEXT DEFAULT 'ONLINE',
    battery_capacity_kwh NUMERIC(4,1) DEFAULT 10.0,
    current_soc_pct    NUMERIC(4,1) DEFAULT 50.0,
    metadata           JSONB DEFAULT '{}'
);

-- Dispatch commands: 2-state lifecycle (SCHEDULED → EXECUTED or FAILED)
-- Written in bulk by the daily scheduler, read by each gateway, updated on completion
CREATE TABLE IF NOT EXISTS dispatch_commands (
    command_id         BIGSERIAL PRIMARY KEY,
    gateway_id         TEXT NOT NULL REFERENCES devices(gateway_id),
    delivery_start     TIMESTAMPTZ NOT NULL,
    delivery_end       TIMESTAMPTZ NOT NULL,
    action             TEXT NOT NULL,
    target_power_kw    NUMERIC(5,2),
    target_soc_pct     NUMERIC(4,1),
    price_eur_mwh      NUMERIC(8,2),
    price_zone         TEXT,
    state              TEXT DEFAULT 'SCHEDULED',
    created_at         TIMESTAMPTZ DEFAULT now(),
    picked_up_at       TIMESTAMPTZ,
    completed_at       TIMESTAMPTZ,
    actual_power_kw    NUMERIC(5,2),
    actual_soc_pct     NUMERIC(4,1),
    latency_ms         INT,
    error_code         TEXT
);

-- Command execution log: one entry per completed command (append-only → replicated to Snowflake)
CREATE TABLE IF NOT EXISTS command_execution_log (
    log_id             BIGSERIAL PRIMARY KEY,
    command_id         BIGINT NOT NULL,
    gateway_id         TEXT NOT NULL,
    event_time         TIMESTAMPTZ DEFAULT now(),
    action             TEXT NOT NULL,
    final_state        TEXT NOT NULL,
    target_power_kw    NUMERIC(5,2),
    actual_power_kw    NUMERIC(5,2),
    actual_soc_pct     NUMERIC(4,1),
    price_eur_mwh      NUMERIC(8,2),
    price_zone         TEXT,
    latency_ms         INT,
    error_code         TEXT
);

-- Dispatch alerts: SLA violations detected by the monitoring process
CREATE TABLE IF NOT EXISTS dispatch_alerts (
    alert_id           BIGSERIAL PRIMARY KEY,
    gateway_id         TEXT,
    alert_type         TEXT,
    severity           TEXT DEFAULT 'INFO',
    created_at         TIMESTAMPTZ DEFAULT now(),
    resolved_at        TIMESTAMPTZ,
    details            JSONB DEFAULT '{}'
);

-- Indexes for the hot query paths
CREATE INDEX IF NOT EXISTS idx_commands_gateway_scheduled ON dispatch_commands(gateway_id, delivery_start) WHERE state = 'SCHEDULED';
CREATE INDEX IF NOT EXISTS idx_commands_stale ON dispatch_commands(created_at) WHERE state = 'SCHEDULED';
CREATE INDEX IF NOT EXISTS idx_exec_log_time ON command_execution_log(event_time);
CREATE INDEX IF NOT EXISTS idx_alerts_open ON dispatch_alerts(alert_type) WHERE resolved_at IS NULL;

$$;

## 4. Seed Dispatch Data

We generate **60 days** of realistic dispatch history, aligned with the same day-ahead prices used by Module 1's VPP telemetry. The scheduler uses EPOWER's price-reactive battery strategy:

| Price Zone | Action | Target Power | Success Rate |
|-----------|--------|--------------|-------------|
| NEGATIVE (< €0/MWh) | MAX_CHARGE | +3.5 to +5.0 kW | 97% |
| LOW (< P25) | CHARGE | +2.0 to +4.5 kW | 96% |
| MEDIUM (P25–P75) | IDLE | 0 kW | 99% |
| HIGH (> P75) | DISCHARGE | -2.0 to -5.0 kW | 95% |

We sample 200 devices per hour (from the full 4,050 fleet) to keep data generation fast while maintaining statistical significance. Each command produces **one log entry** when the device reports its result.

&nbsp;

> **Data volume:** ~200 devices × 24 hours × 60 days = **~288K command records** + **~288K execution log entries** (one per completed command).

In [ ]:
# Seed the Postgres device registry from the existing EPULSE_DEVICES table in Snowflake
devices_df = session.sql("""
    SELECT gateway_id, customer_key
    FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES
    WHERE is_vpp_enrolled = TRUE
""").collect()

print(f"Found {len(devices_df)} VPP-enrolled devices to register in Postgres")

import random
firmware_versions = ['2.3.8', '2.4.0', '2.4.1', '2.4.2', '2.5.0-beta']
battery_capacities = [5.0, 7.5, 10.0, 10.0, 10.0, 12.5, 15.0]

values_list = []
for row in devices_df:
    gw = row['GATEWAY_ID']
    ck = row['CUSTOMER_KEY']
    fw = random.choice(firmware_versions)
    cap = random.choice(battery_capacities)
    soc = round(random.uniform(30, 80), 1)
    status = random.choices(['ONLINE', 'OFFLINE', 'MAINTENANCE'], weights=[95, 4, 1])[0]
    values_list.append(
        f"('{gw}', {ck}, '{fw}', now() - interval '{random.randint(0,300)} seconds', "
        f"'{status}', {cap}, {soc}, '{{}}')"
    )

batch_size = 500
for i in range(0, len(values_list), batch_size):
    batch = values_list[i:i+batch_size]
    insert_sql = f"""
        INSERT INTO devices (gateway_id, customer_key, firmware_version, last_seen_at, status, battery_capacity_kwh, current_soc_pct, metadata)
        VALUES {', '.join(batch)}
        ON CONFLICT (gateway_id) DO NOTHING;
    """
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $${insert_sql}$$").collect()

print(f"✓ Registered {len(devices_df)} devices in Postgres")

In [ ]:
# Generate 60 days of dispatch commands + execution logs using real prices
from collections import defaultdict

prices_df = session.sql("""
    SELECT hour, price_eur_mwh,
           CASE
               WHEN price_eur_mwh < 0 THEN 'NEGATIVE'
               WHEN price_eur_mwh < 88 THEN 'LOW'
               WHEN price_eur_mwh > 144 THEN 'HIGH'
               ELSE 'MEDIUM'
           END AS price_zone
    FROM EPOWER_DEMO.EPOWER_GOLD.MART_DAY_AHEAD_PRICES
    WHERE hour >= DATEADD('day', -60, CURRENT_DATE())
    ORDER BY hour
""").collect()

print(f"Loaded {len(prices_df)} hourly price records")

gateways = [row['GATEWAY_ID'] for row in devices_df]
action_map = {'NEGATIVE': 'MAX_CHARGE', 'LOW': 'CHARGE', 'MEDIUM': 'IDLE', 'HIGH': 'DISCHARGE'}
power_map = {'NEGATIVE': (3.5, 5.0), 'LOW': (2.0, 4.5), 'MEDIUM': (0.0, 0.0), 'HIGH': (-5.0, -2.0)}
success_rate = {'NEGATIVE': 0.97, 'LOW': 0.96, 'MEDIUM': 0.99, 'HIGH': 0.95}

prices_by_date = defaultdict(list)
for p in prices_df:
    prices_by_date[str(p['HOUR'])[:10]].append(p)

total_commands = 0
total_logs = 0
sample_size = 200

print(f"Processing {len(prices_by_date)} days × {sample_size} devices/hour...")

for day_idx, (date_str, day_prices) in enumerate(sorted(prices_by_date.items())):
    if day_idx % 10 == 0:
        print(f"  Day {day_idx+1}/{len(prices_by_date)}: {date_str}")

    sampled_gateways = random.sample(gateways, min(sample_size, len(gateways)))
    cmd_values = []
    log_values = []

    for price_row in day_prices:
        hour_ts = str(price_row['HOUR'])
        price = float(price_row['PRICE_EUR_MWH'])
        zone = price_row['PRICE_ZONE']
        action = action_map[zone]
        pwr_range = power_map[zone]

        for gw in sampled_gateways:
            target_power = round(random.uniform(pwr_range[0], pwr_range[1]), 2) if pwr_range != (0.0, 0.0) else 0.0
            target_soc = round(random.uniform(40, 90), 1)
            succeeded = random.random() < success_rate[zone]
            final_state = 'EXECUTED' if succeeded else 'FAILED'
            latency = random.randint(50, 800) if succeeded else random.randint(5000, 30000)
            actual_power = round(target_power * random.uniform(0.85, 1.05), 2) if succeeded else 0.0
            actual_soc = round(target_soc * random.uniform(0.9, 1.1), 1) if succeeded else target_soc
            error_code = "NULL" if succeeded else f"'TIMEOUT'"

            # Command record (state already set to final — simulating completed history)
            cmd_values.append(
                f"('{gw}', '{hour_ts}'::timestamptz, '{hour_ts}'::timestamptz + interval '1 hour', "
                f"'{action}', {target_power}, {target_soc}, {price}, '{zone}', '{final_state}', "
                f"'{hour_ts}'::timestamptz - interval '5 minutes', "
                f"'{hour_ts}'::timestamptz, "
                f"'{hour_ts}'::timestamptz + interval '{random.randint(1,50)} minutes', "
                f"{actual_power}, {actual_soc}, {latency}, {error_code})"
            )

            # One log entry per completed command (append-only audit)
            log_values.append(
                f"('{gw}', '{hour_ts}'::timestamptz + interval '{random.randint(60,3000)} seconds', "
                f"'{action}', '{final_state}', {target_power}, {actual_power}, {actual_soc}, "
                f"{price}, '{zone}', {latency}, {error_code})"
            )

            total_commands += 1
            total_logs += 1

    # Bulk insert commands
    for i in range(0, len(cmd_values), 500):
        batch = cmd_values[i:i+500]
        sql = f"INSERT INTO dispatch_commands (gateway_id, delivery_start, delivery_end, action, target_power_kw, target_soc_pct, price_eur_mwh, price_zone, state, created_at, picked_up_at, completed_at, actual_power_kw, actual_soc_pct, latency_ms, error_code) VALUES {', '.join(batch)};"
        session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $${sql}$$").collect()

    # Bulk insert execution log (one entry per command)
    for i in range(0, len(log_values), 500):
        batch = log_values[i:i+500]
        sql = f"INSERT INTO command_execution_log (gateway_id, event_time, action, final_state, target_power_kw, actual_power_kw, actual_soc_pct, price_eur_mwh, price_zone, latency_ms, error_code) VALUES {', '.join(batch)};"
        session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $${sql}$$").collect()

print(f"\n✓ Generated {total_commands:,} dispatch commands")
print(f"✓ Generated {total_logs:,} execution log entries (1:1 with commands)")

In [ ]:
# Generate dispatch alerts
alert_types = [('DEVICE_OFFLINE', 'WARNING'), ('COMMAND_TIMEOUT', 'CRITICAL'), ('SOC_DRIFT', 'INFO'), ('COMM_FAILURE', 'WARNING')]
alert_values = []

for _ in range(500):
    gw = random.choice(gateways)
    alert_type, severity = random.choice(alert_types)
    hours_ago = random.randint(1, 60*24)
    resolved = random.random() < 0.85
    resolved_clause = f"now() - interval '{hours_ago - random.randint(1, min(hours_ago, 48))} hours'" if resolved else "NULL"
    alert_values.append(
        f"('{gw}', '{alert_type}', '{severity}', "
        f"now() - interval '{hours_ago} hours', {resolved_clause}, '{{}}')"
    )

for i in range(0, len(alert_values), 100):
    batch = alert_values[i:i+100]
    sql = f"INSERT INTO dispatch_alerts (gateway_id, alert_type, severity, created_at, resolved_at, details) VALUES {', '.join(batch)};"
    session.sql(f"EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $${sql}$$").collect()

print(f"✓ Generated 500 dispatch alerts")

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$
    SELECT 'devices' AS table_name, count(*) AS rows FROM devices
    UNION ALL SELECT 'dispatch_commands', count(*) FROM dispatch_commands
    UNION ALL SELECT 'command_execution_log', count(*) FROM command_execution_log
    UNION ALL SELECT 'dispatch_alerts', count(*) FROM dispatch_alerts
    ORDER BY table_name;
$$;

## 5. pg_lake + Iceberg Sync

Now we set up the **zero-ETL pipeline** from Postgres to Snowflake. The `pg_lake` extension lets Postgres create and manage Iceberg tables on cloud object storage. Combined with `pg_incremental`, we get automated, exactly-once data sync — no external tooling required.

```
command_execution_log (heap) ──pg_incremental──► command_execution_log_iceberg (Iceberg)
                                                         │
                                                    Object Storage (cheap)
                                                         │
                                              Snowflake Catalog Integration
                                                         │
                                                         ▼
                                              VPP_COMMAND_EXECUTION_LOG
                                                  (Snowflake Iceberg Table)
```

&nbsp;

> **Key insight:** We only replicate the `command_execution_log` — it's append-only (one entry per completed command) and time-partitioned. The mutable operational tables (`devices`, `dispatch_commands`) stay in Postgres for transactional access. This also solves the **storage cost concern**: Postgres holds only the hot operational window; the full history lives in cheap Iceberg/object storage, readable by Snowflake.

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$

CREATE EXTENSION IF NOT EXISTS pg_lake CASCADE;
CREATE EXTENSION IF NOT EXISTS pg_cron;
CREATE EXTENSION IF NOT EXISTS pg_incremental;

CREATE TABLE IF NOT EXISTS command_execution_log_iceberg (
    log_id             BIGINT,
    command_id         BIGINT,
    gateway_id         TEXT,
    event_time         TIMESTAMPTZ,
    action             TEXT,
    final_state        TEXT,
    target_power_kw    NUMERIC(5,2),
    actual_power_kw    NUMERIC(5,2),
    actual_soc_pct     NUMERIC(4,1),
    price_eur_mwh      NUMERIC(8,2),
    price_zone         TEXT,
    latency_ms         INT,
    error_code         TEXT
) USING iceberg;

$$;

In [ ]:
%%sql
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$

SELECT incremental.create_time_interval_pipeline(
    pipeline_name      := 'sync_dispatch_log_to_iceberg',
    time_interval      := '1 minute',
    source_table_name  := 'command_execution_log',
    start_time         := (SELECT min(event_time) FROM command_execution_log),
    command            := $inner$
        INSERT INTO command_execution_log_iceberg
        SELECT log_id, command_id, gateway_id, event_time,
               action, final_state, target_power_kw, actual_power_kw,
               actual_soc_pct, price_eur_mwh, price_zone, latency_ms, error_code
        FROM command_execution_log
        WHERE event_time >= $1 AND event_time < $2
    $inner$
);

$$;

In [ ]:
%%sql
-- Verify the initial backfill completed (both counts should match)
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$
    SELECT
        (SELECT count(*) FROM command_execution_log) AS heap_rows,
        (SELECT count(*) FROM command_execution_log_iceberg) AS iceberg_rows;
$$;

## 6. Snowflake Catalog Integration

Now we connect Snowflake to the Postgres-managed Iceberg table. This creates a **zero-copy read path** — Snowflake reads Iceberg metadata and data files directly from object storage, managed by Postgres as the catalog.

| Component | Purpose |
|-----------|--------|
| **Catalog Integration** | Connects Snowflake to Postgres's Iceberg catalog |
| **Iceberg Table** | Snowflake-side reference to the Iceberg data |
| **Auto-Refresh** | Polls for new Iceberg snapshots (every 30s) |

&nbsp;

> **Zero-ETL:** No data pipelines, no Kafka, no Fivetran. Postgres writes to Iceberg via pg_lake, Snowflake reads from Iceberg via catalog integration. The data never leaves object storage.

In [ ]:
%%sql
CREATE OR REPLACE CATALOG INTEGRATION EPOWER_DEMO.EPOWER_OPS.EPULSE_POSTGRES_CATALOG
  CATALOG_SOURCE    = SNOWFLAKE_POSTGRES
  TABLE_FORMAT      = ICEBERG
  CATALOG_NAMESPACE = 'public'
  REST_CONFIG = (
    POSTGRES_INSTANCE      = 'EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH'
    CATALOG_NAME           = 'postgres'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  ENABLED = TRUE;

In [ ]:
%%sql
CREATE OR REPLACE ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG
    CATALOG = 'EPOWER_DEMO.EPOWER_OPS.EPULSE_POSTGRES_CATALOG'
    CATALOG_TABLE_NAME = 'command_execution_log_iceberg';

ALTER ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG
    SET AUTO_REFRESH = TRUE;

In [ ]:
%%sql
-- Query Iceberg table from Snowflake — data flows from Postgres!
SELECT
    count(*) AS total_events,
    count(DISTINCT gateway_id) AS unique_devices,
    min(event_time) AS earliest_event,
    max(event_time) AS latest_event,
    count(CASE WHEN final_state = 'EXECUTED' THEN 1 END) AS successful,
    count(CASE WHEN final_state = 'FAILED' THEN 1 END) AS failed
FROM EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG;

## 7. Analytics Models (Bronze → Gold)

We create two Gold-layer analytics tables from the Postgres-sourced execution log, enriched with device and customer context from Module 1:

| Model | Description | Key Metrics |
|-------|-------------|------------|
| `MART_VPP_DISPATCH_PERFORMANCE` | Hourly dispatch KPIs by region | Success rate, avg/p95 latency, active devices |
| `MART_VPP_FLEET_HEALTH` | Per-device lifetime metrics | Device-level success rate, staleness, avg power |

&nbsp;

> **Note:** In production, these would be dbt models in the existing `epower_dbt/` project. For this module, we create them as CTAS for self-contained simplicity.

In [ ]:
%%sql
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.MART_VPP_DISPATCH_PERFORMANCE AS
WITH events AS (
    SELECT
        DATE_TRUNC('HOUR', event_time) AS hour,
        gateway_id,
        final_state,
        actual_power_kw,
        latency_ms
    FROM EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG
),
device_context AS (
    SELECT d.gateway_id, c.state AS region
    FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES d
    JOIN EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM c ON d.customer_key = c.customer_key
)
SELECT
    e.hour,
    dc.region,
    COUNT(*) AS total_commands,
    COUNT(CASE WHEN e.final_state = 'EXECUTED' THEN 1 END) AS successful_commands,
    COUNT(CASE WHEN e.final_state = 'FAILED' THEN 1 END) AS failed_commands,
    ROUND(COUNT(CASE WHEN e.final_state = 'EXECUTED' THEN 1 END)::FLOAT / NULLIF(COUNT(*), 0) * 100, 2) AS success_rate_pct,
    ROUND(AVG(e.latency_ms), 0) AS avg_latency_ms,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY e.latency_ms), 0) AS p95_latency_ms,
    COUNT(DISTINCT e.gateway_id) AS active_devices,
    ROUND(AVG(ABS(e.actual_power_kw)), 2) AS avg_power_kw
FROM events e
JOIN device_context dc ON e.gateway_id = dc.gateway_id
GROUP BY e.hour, dc.region
ORDER BY e.hour DESC, dc.region;

In [ ]:
%%sql
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.MART_VPP_FLEET_HEALTH AS
WITH device_stats AS (
    SELECT
        gateway_id,
        COUNT(*) AS total_events,
        COUNT(CASE WHEN final_state = 'EXECUTED' THEN 1 END) AS successful,
        COUNT(CASE WHEN final_state = 'FAILED' THEN 1 END) AS failed,
        ROUND(AVG(latency_ms), 0) AS avg_latency_ms,
        MAX(event_time) AS last_activity,
        ROUND(AVG(ABS(actual_power_kw)), 2) AS avg_power_kw
    FROM EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG
    GROUP BY gateway_id
),
device_context AS (
    SELECT d.gateway_id, d.customer_key, c.customer_name, c.city, c.state AS region, c.customer_type
    FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES d
    JOIN EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM c ON d.customer_key = c.customer_key
    WHERE d.is_vpp_enrolled = TRUE
)
SELECT
    dc.gateway_id,
    dc.customer_name,
    dc.city,
    dc.region,
    dc.customer_type,
    ds.total_events,
    ds.successful,
    ds.failed,
    ROUND(ds.successful::FLOAT / NULLIF(ds.total_events, 0) * 100, 2) AS success_rate_pct,
    ds.avg_latency_ms,
    ds.avg_power_kw,
    ds.last_activity,
    DATEDIFF('hour', ds.last_activity, CURRENT_TIMESTAMP()) AS hours_since_last_activity
FROM device_context dc
LEFT JOIN device_stats ds ON dc.gateway_id = ds.gateway_id
ORDER BY ds.success_rate_pct ASC NULLS LAST;

In [ ]:
%%sql
SELECT 'MART_VPP_DISPATCH_PERFORMANCE' AS table_name, COUNT(*) AS rows FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_DISPATCH_PERFORMANCE
UNION ALL SELECT 'MART_VPP_FLEET_HEALTH', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_FLEET_HEALTH;

## 8. Semantic View + Agent Update

We create a **VPP_DISPATCH_SEMANTIC_VIEW** that exposes the dispatch performance data to the Intelligence Agent, then update the agent with a new `dispatch_analyst` tool.

| Metric | Description |
|--------|------------|
| `success_rate_pct` | % of commands that reached EXECUTED state |
| `avg_latency_ms` | Mean time from DISPATCHED to EXECUTED/FAILED |
| `p95_latency_ms` | 95th percentile latency (SLA indicator) |
| `total_commands` | Hourly command volume per region |
| `active_devices` | Devices that responded in each hour |

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.VPP_DISPATCH_SEMANTIC_VIEW
  COMMENT = 'VPP dispatch command performance — sourced from Snowflake Postgres via pg_lake'
AS
TABLES (
  DISPATCH REFERENCES EPOWER_DEMO.EPOWER_GOLD.MART_VPP_DISPATCH_PERFORMANCE (
    PRIMARY KEY (HOUR, REGION)
    FACTS (
      TOTAL_COMMANDS COMMENT 'Number of dispatch commands issued',
      SUCCESSFUL_COMMANDS COMMENT 'Commands that reached EXECUTED state',
      FAILED_COMMANDS COMMENT 'Commands that ended in FAILED state',
      SUCCESS_RATE_PCT COMMENT 'Percentage of successful commands (0-100)'
        SYNONYMS ('success rate', 'Erfolgsrate', 'reliability', 'Zuverlässigkeit'),
      AVG_LATENCY_MS COMMENT 'Average command latency in milliseconds'
        SYNONYMS ('latency', 'Latenz', 'response time', 'Antwortzeit'),
      P95_LATENCY_MS COMMENT '95th percentile latency in milliseconds (SLA metric)'
        SYNONYMS ('p95', 'SLA latency', 'tail latency'),
      ACTIVE_DEVICES COMMENT 'Number of devices that responded in this hour'
        SYNONYMS ('responding devices', 'online devices'),
      AVG_POWER_KW COMMENT 'Average absolute power delivered per command in kW'
    )
    DIMENSIONS (
      HOUR COMMENT 'Hourly time bucket (UTC)'
        SYNONYMS ('time', 'timestamp', 'date', 'Zeitpunkt', 'Stunde'),
      REGION COMMENT 'German geographic region (Nord, Süd, West, Ost)'
        SYNONYMS ('region', 'area', 'Gebiet', 'Bundesland')
    )
  ),
  FLEET REFERENCES EPOWER_DEMO.EPOWER_GOLD.MART_VPP_FLEET_HEALTH (
    PRIMARY KEY (GATEWAY_ID)
    FACTS (
      TOTAL_EVENTS COMMENT 'Total command execution events for this device',
      SUCCESSFUL COMMENT 'Count of successful executions',
      FAILED COMMENT 'Count of failed executions',
      SUCCESS_RATE_PCT COMMENT 'Device-level success rate percentage',
      AVG_LATENCY_MS COMMENT 'Device-level average latency',
      AVG_POWER_KW COMMENT 'Average power delivered by this device',
      HOURS_SINCE_LAST_ACTIVITY COMMENT 'Hours since last command execution'
        SYNONYMS ('inactive hours', 'offline duration', 'last seen')
    )
    DIMENSIONS (
      GATEWAY_ID COMMENT 'Unique device/gateway identifier'
        SYNONYMS ('device', 'gateway', 'Gerät'),
      CUSTOMER_NAME COMMENT 'Customer who owns this device',
      CITY COMMENT 'City where device is installed',
      REGION COMMENT 'Geographic region',
      CUSTOMER_TYPE COMMENT 'Customer segment (Privatkunde, Kleingewerbe, Gewerbekunde)'
    )
  )
)
RELATIONSHIPS (
  DISPATCH(REGION) REFERENCES FLEET(REGION)
);

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, VPP dispatch command performance, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export, IoT → vpp_telemetry_analyst
    - VPP dispatch commands, success rate, latency, device health, fleet status → dispatch_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices, spot prices, energy market"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, heatpump, grid import/export"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: dispatch_analyst, description: "VPP dispatch command performance: success rates, latency, device fleet health, command failures. Data from Snowflake Postgres via pg_lake."}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  dispatch_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.VPP_DISPATCH_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## 9. Verification & Demo Questions

### Pipeline Summary

| Layer | Object | Source |
|-------|--------|--------|
| **Postgres** | `command_execution_log` | OLTP dispatch engine |
| **Iceberg** | `command_execution_log_iceberg` | pg_lake managed |
| **Bronze** | `VPP_COMMAND_EXECUTION_LOG` | Catalog integration (auto-refresh) |
| **Gold** | `MART_VPP_DISPATCH_PERFORMANCE` | Hourly aggregation |
| **Gold** | `MART_VPP_FLEET_HEALTH` | Per-device metrics |
| **Semantic View** | `VPP_DISPATCH_SEMANTIC_VIEW` | Text-to-SQL access |
| **Agent** | `EPOWER_AGENT` | Now with `dispatch_analyst` tool (13 tools total) |

In [ ]:
%%sql
-- Sample: Hourly dispatch performance
SELECT hour, region, total_commands, success_rate_pct, avg_latency_ms, p95_latency_ms, active_devices
FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_DISPATCH_PERFORMANCE
ORDER BY hour DESC
LIMIT 20;

In [ ]:
%%sql
-- Devices with lowest success rates (fleet health issues)
SELECT gateway_id, customer_name, city, region, total_events, success_rate_pct, avg_latency_ms, hours_since_last_activity
FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_FLEET_HEALTH
WHERE total_events > 10
ORDER BY success_rate_pct ASC
LIMIT 10;

### Demo Questions for the Agent

Try these in Snowflake Intelligence — they exercise the new `dispatch_analyst` tool:

| # | Question | What it tests |
|---|----------|---------------|
| 1 | *"What's the VPP dispatch success rate this week?"* | Basic dispatch KPI |
| 2 | *"Which region has the highest command failure rate?"* | Regional comparison |
| 3 | *"Show me the average command latency by region over the last 7 days"* | Time-series + regional |
| 4 | *"Are there any devices with a success rate below 90%?"* | Fleet health drill-down |
| 5 | *"Wie ist die p95 Latenz der Dispatch-Befehle in der Region Nord?"* | German, specific SLA metric |
| 6 | *"Compare dispatch success rate with actual telemetry export — are we delivering what we promised?"* | **Cross-tool**: dispatch + telemetry |

&nbsp;

> **Presenter tip:** Question 6 forces the agent to combine data from **two sources** (Postgres-sourced dispatch + Snowflake-native telemetry). This demonstrates the value of the unified platform.

### Live Demo: Real-Time Sync

To demonstrate the **near real-time** pipeline during a live demo:

1. **Insert new data in Postgres** (simulates a fresh dispatch cycle)
2. **Wait 30–60 seconds** (auto-refresh interval)
3. **Query in Snowflake** (new data appears automatically)

In [ ]:
%%sql
-- Simulate devices completing commands RIGHT NOW in Postgres
EXECUTE POSTGRES EPOWER_DEMO.EPOWER_OPS.EPULSE_DISPATCH $$
    INSERT INTO command_execution_log (gateway_id, event_time, action, final_state, target_power_kw, actual_power_kw, actual_soc_pct, price_eur_mwh, price_zone, latency_ms, error_code)
    SELECT
        gateway_id,
        now(),
        'DISCHARGE',
        CASE WHEN random() < 0.96 THEN 'EXECUTED' ELSE 'FAILED' END,
        ROUND((random() * 3 + 2)::numeric, 2),
        ROUND((random() * 3 + 1.5)::numeric, 2),
        ROUND((random() * 50 + 30)::numeric, 1),
        ROUND((random() * 100 + 80)::numeric, 2),
        'HIGH',
        (random() * 500 + 50)::int,
        NULL
    FROM devices
    ORDER BY random()
    LIMIT 100;
$$;
-- Data flows: Postgres → pg_incremental → Iceberg → auto-refresh → Snowflake
-- Wait ~60 seconds, then run the next cell

In [ ]:
%%sql
-- Verify real-time data arrival (run ~60s after previous cell)
SELECT
    MAX(event_time) AS most_recent_event,
    DATEDIFF('second', MAX(event_time), CURRENT_TIMESTAMP()) AS seconds_ago,
    COUNT(*) AS total_rows
FROM EPOWER_DEMO.EPOWER_BRONZE.VPP_COMMAND_EXECUTION_LOG;

---

## Summary

In this module you built a complete **operational-to-analytical pipeline** using Snowflake Postgres:

| What | How |
|------|-----|
| **OLTP dispatch engine** | Snowflake Postgres with devices, commands, execution log |
| **Zero-ETL replication** | pg_lake + pg_incremental → Iceberg → Snowflake (no external tooling) |
| **Near real-time** | Auto-refresh polls every 30 seconds |
| **Analytics** | Gold-layer marts for dispatch performance and fleet health |
| **AI-ready** | Semantic View + Cortex Agent tool — queryable in natural language |

The key architectural insight: **Postgres handles what Snowflake can't** (sub-second OLTP, state machines, high-concurrency device polling), while **Snowflake handles what Postgres can't** (petabyte-scale analytics, ML, natural language access, cross-domain joins with 6 other business domains).

Together, they form a complete platform — connected by open standards (Iceberg), with no middleware required.

---

*EPOWER Module 2 — Snowflake Postgres + pg_lake — Powered by Snowflake*